# IRRBB comprehensive demo

## 1. Load packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from banking_risk.irrbb import (
    # banking book and atomic units
    Position,
    Standard_Banking_Book,
    NMD_Portfolio,
    NMD_Banking_Book,
    # curves
    QL_Curve_Adapter, # to 
    # scenarios
    Scenario_Set,
    # calculators
    SA_EVE_Calculator,
    SA_NII_Calculator,
    Repricing_Gap,
)
from banking_risk.shared.curve_projection import Curve_Projection
from banking_risk.utils.reporting import Dark_Style, Light_Style

## 2. Curve construction

* Build a Zero_Curve from a tenor/rate point list (flat and slightly upward-sloping variants) — show the points property
- Wrap with QL_Curve_Adapter and demonstrate `.discount(t)` and `.zero_rate(t)`
- Plot the base curve shape — spot rates vs. tenor

In [ ]:
tenors_y   = [0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0]
base_rates = [0.032, 0.033, 0.034, 0.035, 0.036, 0.037, 0.038, 0.039, 0.040, 0.040, 0.039]

base_curve = QL_Curve_Adapter.from_arrays(tenors_y, base_rates, "2026-06-01")
proj       = Curve_Projection(base_curve)

In [ ]:
import dataclasses
pd.DataFrame(dataclasses.asdict(proj.irrbb))

## 3. Banking book
- Define a small but representative Standard_Banking_Book (mix of fixed assets, floating assets, fixed liabilities)
- Define one NMD_Portfolio (retail, stable/volatile split) and build an NMD_Banking_Book
- Show `book.total_assets()`, `book.total_liabilities()`, `book.equity()`, `book.balance_check()`

#### 3.1. Example: standard banking book
Only has liabilities with **fixed-contractual** maturity date, as bonds and term deposits.

In [ ]:
# ── Standard banking book ─────────────────────────────────────────────────────
#    Mix: fixed mortgage, floating corporate loan, fixed senior liabilities.
#    Numbers chosen so assets = liabilities + Tier 1 (balance_check passes).

assets = [
    Position("Mortgage_10Y",  "asset", "EUR", 100_000_000, 120,  3, 0.039, False),
    Position("Corp_Loan_3M",  "asset", "EUR",  50_000_000,  60,  3, 0.037, True, 3),
]
liabilities = [
    Position("Retail_Bond_3Y", "liability", "EUR", 80_000_000, 36, 6, 0.033, False),
    Position("Senior_Bond_5Y", "liability", "EUR", 50_000_000, 60, 6, 0.035, False),
]
TIER1 = 20_000_000   # 150M assets − 130M liabilities

book = Standard_Banking_Book(assets + liabilities, tier1_capital=TIER1)

print(f"total assets      : {book.total_assets():>15,.0f}")
print(f"total liabilities : {book.total_liabilities():>15,.0f}")
print(f"equity            : {book.equity():>15,.0f}")
print(f"balance_check     : {book.balance_check()}")



#### 3.2. Example: banking book with NMD
Non-Maturity Deposit (no-contractual maturity). 
- Behavioural maturity: customers do not withdraw all at once. 
- Banks model the behavioural average life based on historical withdrawal.
- A current account is "overnight" (can be withdrawn any time) but has behavioural maturity of 3-5 years (sticky balance)
- Rate stickiness: banks do not pass rate changes through to NMD customers immediately or fully. 
- Passthrough rates are also modelled behaviourally




In [ ]:

# ── NMD portfolio ─────────────────────────────────────────────────────────────
#    Retail savings: 70 % stable (avg repricing 3.5Y, ≤ 5Y EBA/GL/2022/14 cap),
#    1. volatile share is treated as overnight - reprices immediately.
#    2. stable share is treated as fixed-rate - reprices according to avg repricing (capped at 5y for retail per EBA/GL/2022/14).

nmd = NMD_Portfolio(
    name="Retail_Savings",
    currency="EUR",
    volume=50_000_000,
    nmd_type="retail",
    stable_ratio=0.70,
    stable_avg_repricing_years=3.5,
    rate=0.010,
)
print(f"\nNMD stable   : {nmd.stable_volume:>12,.0f}  (repricing avg {nmd.stable_avg_repricing_years}Y)")
print(f"NMD volatile : {nmd.volatile_volume:>12,.0f}  (overnight proxy)")

In [ ]:
# ── NMD banking book ──────────────────────────────────────────────────────────
#    Contractual liabilities reduced by NMD volume so balance still holds:
#    150M assets = 80M bonds + 50M NMD + 20M Tier 1

liabilities = [
    Position("Retail_Bond_3Y", "liability", "EUR", 80_000_000, 36, 6, 0.033, False),
]
nmd_book = NMD_Banking_Book(
    positions=assets + liabilities,
    nmd_portfolios=[nmd],
    tier1_capital=TIER1,
)

print(f"\nnmd_book total assets      : {nmd_book.total_assets():>15,.0f}")
print(f"nmd_book total liabilities : {nmd_book.total_liabilities():>15,.0f}")
print(f"nmd_book balance_check     : {nmd_book.balance_check()}")
print(f"nmd_book position count    : {len(nmd_book.positions())}  (3 contractual + 2 synthetic NMD)")
print(f"The NMD position is divided into the stable and volatile components, each  treated as a separate position.")

## 4. Scenarios
- Instantiate all six EBA named scenarios (Parallel_Up, etc.) and a Scenario_Set
- Show scenario.shocked_curve(base_curve) for Parallel_Up — plot base vs. shocked curve on same axes
- Plot all six shocked curves overlaid on the base curve (one chart)

In [ ]:
scenario_set = Scenario_Set("EUR")

print(f"Scenarios loaded for {scenario_set.currency}:")
for s in scenario_set:
    print(f"  {s.name}")

In [ ]:
Dark_Style().apply()
p      = Dark_Style().palette
colors = [p["cyan"], p["red"], p["green"], p["amber"], p["blue"], p["purple"]]

fig, ax = plt.subplots()
ax.plot(proj.plot.vertices, proj.plot.rates * 100,
        color=p["text_muted"], lw=1.5, ls="--", label="base")

for scenario, color in zip(scenario_set, colors):
    shocked = scenario.shock(proj.plot.rates, proj.plot.vertices)
    ax.plot(proj.plot.vertices, shocked * 100, color=color, lw=1.5, label=scenario.name)

ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Zero rate (%)")
ax.set_title("EBA supervisory shock scenarios — EUR (EBA/RTS/2022/10 Annex III)")
ax.legend(fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

## 5. Repricing gap

- Repricing_Gap(book).compute() → DataFrame
- Bar chart: assets vs. liabilities per EBA bucket, gap line

In [ ]:
gap = Repricing_Gap(book)
gap.to_table()


In [ ]:
gap.plot()

### 6. EVE SOT

- `SA_EVE_Calculator().compute(book, base_curve, scenario_set)` → EVE_Result
- Thin API: `result.to_table()`, `result.plot()`
- Explicit style: `EVE_Reporter(Light_Style()).plot(result)` 
— show both Dark_Style and Light_Style side by side


In [ ]:
eve_result = SA_EVE_Calculator().compute(
    book=book,
    scenarios={"EUR": scenario_set},
    curves={"EUR": base_curve},
)

print(f"worst scenario  : {eve_result.worst_scenario}")
print(f"ΔEVE            : {eve_result.worst_delta_eve:>12,.0f}")
print(f"Tier 1          : {eve_result.tier1_capital:>12,.0f}")
print(f"SOT ratio       : {eve_result.sot_ratio:.2%}")
print(f"outlier         : {eve_result.is_outlier}")


In [ ]:
eve_result.to_table()


In [ ]:
eve_result.plot()


### 7. NII SOT

- `SA_NII_Calculator().compute(book, base_curve, scenario_set)` → NII_Result
- Same pattern: `result.to_table()`, `result.plot()`, explicit `NII_Reporter(Dark_Style()).plot(result)`
- Call out the 5% Tier 1 SOT threshold in the table


In [ ]:
nii_result = SA_NII_Calculator().compute(
    book=book,
    scenarios={"EUR": scenario_set},
    curves={"EUR": base_curve},
)

print(f"ΔNII parallel_up   : {nii_result.delta_nii['parallel_up']:>12,.0f}")
print(f"ΔNII parallel_down : {nii_result.delta_nii['parallel_down']:>12,.0f}")
print(f"SOT ratio          : {nii_result.sot_ratio:.2%}  (threshold 5 %)")
print(f"outlier            : {nii_result.is_outlier}")


In [ ]:
nii_result.to_table()


In [ ]:
nii_result.plot()


8. NMD variant

- Repeat EVE and NII with the NMD_Banking_Book — side-by-side comparison of SOT ratios with and without NMD modelling to show the behavioural assumption effect


In [ ]:
eve_nmd = SA_EVE_Calculator().compute(
    book=nmd_book,
    scenarios={"EUR": scenario_set},
    curves={"EUR": base_curve},
)

nii_nmd = SA_NII_Calculator().compute(
    book=nmd_book,
    scenarios={"EUR": scenario_set},
    curves={"EUR": base_curve},
)

In [ ]:
comparison = pd.DataFrame(
    {
        "Standard book": {
            "EVE worst scenario": eve_result.worst_scenario,
            "ΔEVE":               f"{eve_result.worst_delta_eve:,.0f}",
            "EVE SOT ratio":      f"{eve_result.sot_ratio:.2%}",
            "EVE outlier":        eve_result.is_outlier,
            "NII SOT ratio":      f"{nii_result.sot_ratio:.2%}",
            "NII outlier":        nii_result.is_outlier,
        },
        "NMD book": {
            "EVE worst scenario": eve_nmd.worst_scenario,
            "ΔEVE":               f"{eve_nmd.worst_delta_eve:,.0f}",
            "EVE SOT ratio":      f"{eve_nmd.sot_ratio:.2%}",
            "EVE outlier":        eve_nmd.is_outlier,
            "NII SOT ratio":      f"{nii_nmd.sot_ratio:.2%}",
            "NII outlier":        nii_nmd.is_outlier,
        },
    }
)
comparison
